# A-5 WS2 — Llama-3.1-8B bf16 harvest via Undercurrent, on Modal

**Before Cell 1:** click the key icon (Colab secrets) and add, with notebook access ON:
`MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`, `HF_TOKEN`.
Then upload `a5_modal_harvest_v2.py` to the Files pane (`/content`).

Run cells in order. Each prints **PASS** or stops with the reason.

In [ ]:
# CELL 1 — install the Modal client.  ~30 s
!pip install -q "modal>=1.0"
import modal, subprocess
print("modal", modal.__version__)
print("PASS")

In [ ]:
# CELL 2 — authenticate Modal from Colab secrets (never printed).  ~10 s
import subprocess
from google.colab import userdata

def secret(name):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        raise SystemExit(f"Add {name} in Colab secrets (key icon), enable notebook access, re-run.")
    return value

r = subprocess.run(["modal", "token", "set",
                    "--token-id", secret("MODAL_TOKEN_ID"),
                    "--token-secret", secret("MODAL_TOKEN_SECRET")],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("modal token set failed:\n" + r.stderr[-800:])
print(subprocess.run(["modal", "profile", "current"], capture_output=True, text=True).stdout.strip())

# HF token as a Modal secret, so containers can download gated Llama.
r = subprocess.run(["modal", "secret", "create", "huggingface",
                    f"HF_TOKEN={secret('HF_TOKEN')}", "--force"],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("modal secret create failed:\n" + r.stderr[-800:])
print("HF secret 'huggingface' set")
print("PASS")

In [ ]:
# CELL 3 — mount Drive, (re)upload harvest code + data to the a5-data volume.  ~1-3 min
import shutil, subprocess
from pathlib import Path
from google.colab import drive
drive.mount("/content/gdrive")

ROOT = next((p for p in [
    Path("/content/gdrive/MyDrive/MYSSP/a5-artifacts"),
    Path("/content/gdrive/MyDrive/a5-artifacts"),
] if p.is_dir()), None)
if ROOT is None:
    raise SystemExit("Project root a5-artifacts not found under /content/gdrive/MyDrive.")
H = ROOT / "harvest"
print("project root:", ROOT)

assert Path("/content/a5_modal_harvest_v2.py").is_file(), \
    "Upload a5_modal_harvest_v2.py to /content (Files pane) first."

# Extraction spec: 32 layers at prompt[-1] (A1/A6) + 16 odd layers at prompt[-6] (position split).
layers_all = ", ".join(str(i) for i in range(32))
layers_odd = ", ".join(str(i) for i in range(1, 32, 2))
SPEC = (
    'version: "1"\n\nextraction_points:\n'
    f'  - name: last_prefill\n    layers: [{layers_all}]\n'
    '    tensor_type: residual_stream\n    position: "prompt[-1]"\n'
    '    probe_type: collector\n    probe_kind: single_shot\n\n'
    f'  - name: last_instruction\n    layers: [{layers_odd}]\n'
    '    tensor_type: residual_stream\n    position: "prompt[-6]"\n'
    '    probe_type: collector\n    probe_kind: single_shot\n'
)

stage = Path("/content/a5_stage")
shutil.rmtree(stage, ignore_errors=True)
(stage / "code").mkdir(parents=True)
(stage / "code" / "llama31-8b-promptpos.yaml").write_text(SPEC)

sources = {
    "pairs_table_v1_final-2.csv": ROOT / "Dennis" / "data table" / "pairs_table_v1_final-2.csv",
    "pair_map__d240ec65.parquet": ROOT / "Dennis" / "Activations" /
        "Full harvest Qwen 2.5-7B activations" / "pair_maps" / "pair_map__d240ec65.parquet",
    "code/run.py": H / "run.py",
    "code/collector.py": H / "collector.py",
    "code/drive_archive.py": H / "drive_archive.py",
}

def on_volume(rel):
    folder = str(Path("/") / Path(rel).parent)
    r = subprocess.run(["modal", "volume", "ls", "a5-data", folder], capture_output=True, text=True)
    return Path(rel).name in r.stdout

for rel, src in sources.items():
    if src.is_file():
        shutil.copyfile(src, stage / rel)
        print(f"  staged  {rel:<32} {src.stat().st_size:>12,} bytes")
    elif on_volume(rel):
        print(f"  kept    {rel:<32} (not on Drive, already on volume)")
    else:
        raise SystemExit(f"MISSING: {rel}\n  expected at {src}\n  and not on the volume either.")

for item in sorted(stage.iterdir()):
    r = subprocess.run(["modal", "volume", "put", "a5-data", str(item), f"/{item.name}", "--force"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(f"upload of {item.name} failed:\n{r.stderr[-800:]}")
    print("  uploaded", item.name)

print(subprocess.run(["modal", "volume", "ls", "a5-data", "/code"], capture_output=True, text=True).stdout)
print("PASS")

In [ ]:
# CELL 4 — build the image and verify every input is on the volume.
# First run builds the container image: ~4-7 min. Later runs: ~30 s.
%cd /content
!modal run a5_modal_harvest_v2.py::prepare
# Must end with: "All required inputs present."   Anything under MISSING -> stop.

In [ ]:
# CELL 5 — dry plan: loads + validates the pairs table inside the container.  ~1-2 min
!A5_PROFILE=prompts modal run a5_modal_harvest_v2.py::plan
# Must show: "n_prompts": 18500  and  "n_shards": 37.  If not, stop and paste it to Claude.

In [ ]:
# CELL 6 — THE HARVEST. Llama-3.1-8B, bf16, Undercurrent, 18,500 prompts,
# last_prefill x 32 layers + last_instruction x 16 layers.
# ~3-5 min weight download (first time only), then ~25-45 min for 37 shards on 20 A100s.
# --detach: queued shards keep running on Modal even if this Colab tab disconnects.
# Resumable: re-running skips every shard whose archive already verifies.
!A5_PROFILE=prompts modal run --detach a5_modal_harvest_v2.py::harvest_all
# Success = "Succeeded: 37   Failed: 0" and "Prompts harvested: 18,500".
# If you see "add a payment method" for A100: re-run with  A5_GPU=L4  in front (slower, ~2x).

In [ ]:
# CELL 7 — status (run any time, including while Cell 6 runs from another tab).  ~30 s
!A5_PROFILE=prompts modal run a5_modal_harvest_v2.py::status
# Done when: Archives: 37, Shard manifests: 37, No error logs.